# Employee Data Quality & Human Capital Analytics
## SQL Data Analysis — HRDataset_v14

**Dataset:** Human Resources Data Set (Kaggle)  
**Tools:** Google Colab, Python, Pandas, SQLite  
**Tahap proyek:** 5 — SQL Data Analysis

Notebook ini menggunakan hasil Data Cleaning & Validation dari tahap sebelumnya untuk menganalisis komposisi pegawai, status kepegawaian, perekrutan, kinerja, keterlibatan pegawai, dan absensi menggunakan SQL.

Dataset bersifat **sintetis** dan digunakan untuk pembelajaran. Analisis menggunakan kolom yang sudah tersedia, tanpa menambahkan variabel pada dataset utama.

**Sumber:** [Human Resources Data Set — Kaggle](https://www.kaggle.com/datasets/rhuebner/human-resources-data-set)

## 1. Analysis Objectives

Pertanyaan yang akan dijawab:

1. Bagaimana komposisi pegawai aktif dan terminasi pada setiap departemen?
2. Bagaimana distribusi perekrutan dari waktu ke waktu dan alasan terminasi yang tercatat?
3. Bagaimana distribusi tingkat kinerja antar departemen?
4. Bagaimana perbedaan skor engagement, satisfaction, absensi, dan keterlambatan antar kelompok?
5. Apakah terdapat temuan kualitas data yang masih memerlukan pemeriksaan?

Analisis menggunakan satu tabel `employees` karena dataset sumber hanya memiliki satu tabel pegawai. Pembuatan tabel lain yang mengandung data tambahan tidak diperlukan.

## 2. Load the Cleaned Dataset

Unggah **`HRDataset_v14_clean.csv`** atau **`HR_Analytics_Cleaned_Output.zip`** dari notebook cleaning. ZIP dapat berisi laporan validasi tambahan, tetapi analisis utama hanya memerlukan CSV hasil cleaning.

File diunggah saat notebook dijalankan, sehingga data tidak perlu disimpan permanen di repository GitHub.

In [ ]:
import io
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from google.colab import files

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload satu file: HRDataset_v14_clean.csv atau ZIP hasil cleaning.")

uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
quality_report_previous = None

if uploaded_name.lower().endswith(".zip"):
    with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as archive:
        matches = [
            name for name in archive.namelist()
            if Path(name).name.lower() == "hrdataset_v14_clean.csv"
        ]
        if not matches:
            raise FileNotFoundError("HRDataset_v14_clean.csv tidak ditemukan di ZIP.")
        df = pd.read_csv(io.BytesIO(archive.read(matches[0])))

        reports = [
            name for name in archive.namelist()
            if Path(name).name.lower() == "data_quality_report.csv"
        ]
        if reports:
            quality_report_previous = pd.read_csv(
                io.BytesIO(archive.read(reports[0]))
            )
elif uploaded_name.lower().endswith(".csv"):
    df = pd.read_csv(io.BytesIO(uploaded_bytes))
else:
    raise ValueError("Gunakan file CSV hasil cleaning atau ZIP hasil cleaning.")

print("Dataset dimuat:", df.shape)
display(df.head(3))

Saving HRDataset_v14_clean.csv to HRDataset_v14_clean (1).csv
Dataset dimuat: (311, 36)


,Employee_Name,EmpID,MarriedID,MaritalStatusID,GenderID,EmpStatusID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,...,ManagerName,ManagerID,RecruitmentSource,PerformanceScore,EngagementSurvey,EmpSatisfaction,SpecialProjectsCount,LastPerformanceReview_Date,DaysLateLast30,Absences
0,"Adinolfi, Wilson K",10026,0,0,1,1,5,4,0,62506,...,Michael Albert,22.0,LinkedIn,Exceeds,4.60,5,0,2019-01-17,0,1
1,"Ait Sidi, Karthikeyan",10084,1,1,1,5,3,3,0,104437,...,Simon Roup,4.0,Indeed,Fully Meets,4.96,3,6,2016-02-24,0,17
2,"Akinkuolie, Sarah",10196,1,1,0,5,5,3,0,64955,...,Kissy Sullivan,20.0,LinkedIn,Fully Meets,3.02,3,0,2012-05-15,0,3


### 2.1 Validate the Input

Sebelum memuat data ke database, periksa kolom wajib dan keunikan `EmpID`. Tanggal dikonversi kembali dari CSV ke tipe datetime agar tersimpan dalam format yang konsisten ketika dimasukkan ke SQLite.

Pemeriksaan ini tidak mengubah jumlah record maupun menambahkan kolom pada dataset.

In [ ]:
required_columns = [
    "EmpID", "Department", "EmploymentStatus", "Termd", "DateofHire",
    "DateofTermination", "TermReason", "PerformanceScore",
    "EngagementSurvey", "EmpSatisfaction", "Absences",
    "DaysLateLast30", "ManagerID", "LastPerformanceReview_Date"
]

missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Kolom wajib tidak ditemukan: {missing_columns}")

if df["EmpID"].isna().any() or df["EmpID"].duplicated().any():
    raise ValueError("EmpID harus terisi dan unik sebelum analisis SQL.")

print("Jumlah pegawai unik:", df["EmpID"].nunique())
print("Jumlah atribut:", df.shape[1])

if quality_report_previous is not None:
    print("Laporan validasi dari notebook cleaning tersedia.")
    display(quality_report_previous)
else:
    print("Laporan validasi tidak diunggah; ringkasan lengkap ada di notebook cleaning.")

Jumlah pegawai unik: 311
Jumlah atribut: 36
Laporan validasi tidak diunggah; ringkasan lengkap ada di notebook cleaning.


## 3. Prepare the SQLite Database

SQLite dipilih karena dapat dijalankan langsung di Google Colab tanpa instalasi database server. Seluruh 36 kolom asli dimasukkan ke tabel `employees`.

Kolom tanggal disimpan sebagai teks ISO (`YYYY-MM-DD`) agar fungsi tanggal SQLite dan perbandingan kronologi dapat digunakan dengan konsisten. Konversi angka dipakai hanya untuk kolom yang memang bersifat numerik.

In [ ]:
import sqlite3

# DataFrame terpisah dipakai agar format SQL tidak mengubah hasil cleaning.
df_sql = df.copy()

date_columns = [
    "DOB", "DateofHire", "DateofTermination",
    "LastPerformanceReview_Date"
]

for column in date_columns:
    if column in df_sql.columns:
        parsed = pd.to_datetime(df_sql[column], errors="coerce")
        original_nonmissing = df_sql[column].notna()
        parsing_failed = original_nonmissing & parsed.isna()
        if parsing_failed.any():
            raise ValueError(
                f"Tanggal gagal dibaca pada {column}: {parsing_failed.sum()} record."
            )
        df_sql[column] = parsed.dt.strftime("%Y-%m-%d")

numeric_columns = [
    "Termd", "Salary", "EngagementSurvey", "EmpSatisfaction",
    "Absences", "DaysLateLast30"
]

for column in numeric_columns:
    if column in df_sql.columns:
        df_sql[column] = pd.to_numeric(df_sql[column], errors="raise")

output_dir = Path("/content/hr_sql_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

db_path = output_dir / "hc_analytics.sqlite"
conn = sqlite3.connect(db_path)

df_sql.to_sql("employees", conn, if_exists="replace", index=False)

print("Database:", db_path.name)
print("Rows in employees:", pd.read_sql_query(
    "SELECT COUNT(*) AS row_count FROM employees", conn
).iloc[0, 0])

Database: hc_analytics.sqlite
Rows in employees: 311


### 3.1 Inspect the Database Structure

Pemeriksaan skema membantu memastikan kolom, jumlah record, dan tipe penyimpanan sudah terbaca di SQLite. Pada notebook ini hanya ada satu tabel data utama, yaitu `employees`.

In [ ]:
schema = pd.read_sql_query("PRAGMA table_info(employees);", conn)
display(schema[["name", "type", "notnull"]])

assert pd.read_sql_query(
    "SELECT COUNT(*) AS n FROM employees", conn
)["n"].iloc[0] == len(df), "Jumlah record SQLite berbeda dari CSV."

assert len(schema) == len(df.columns), "Jumlah kolom SQLite berbeda dari CSV."
print("Database integrity check passed.")

,name,type,notnull
0,Employee_Name,TEXT,0
1,EmpID,INTEGER,0
2,MarriedID,INTEGER,0
3,MaritalStatusID,INTEGER,0
4,GenderID,INTEGER,0
5,EmpStatusID,INTEGER,0
6,DeptID,INTEGER,0
7,PerfScoreID,INTEGER,0
8,FromDiversityJobFairID,INTEGER,0
9,Salary,INTEGER,0


Database integrity check passed.


## 4. Workforce Overview

Query pertama merangkum total pegawai, jumlah pegawai yang belum mengalami terminasi, jumlah pegawai yang telah terminasi, serta jumlah departemen. Perhitungan menggunakan `Termd` sebagai penanda status terminasi.

Angka ini menggambarkan komposisi **record dalam dataset**, bukan kondisi kepegawaian organisasi saat ini.

In [ ]:
query_workforce = """
SELECT
    COUNT(DISTINCT EmpID) AS Total_Employees,
    SUM(CASE WHEN Termd = 0 THEN 1 ELSE 0 END) AS Not_Terminated,
    SUM(CASE WHEN Termd = 1 THEN 1 ELSE 0 END) AS Terminated,
    COUNT(DISTINCT Department) AS Departments
FROM employees;
"""

workforce_kpi = pd.read_sql_query(query_workforce, conn)
display(workforce_kpi)

,Total_Employees,Not_Terminated,Terminated,Departments
0,311,207,104,6


### 4.1 Department Composition and Recorded Termination Share

Untuk membandingkan departemen dengan ukuran berbeda, kita menghitung jumlah pegawai, jumlah terminasi, dan proporsi record terminasi pada masing-masing departemen.

`Termination_Share_Pct` merupakan jumlah terminasi dibagi total record dalam departemen. Metrik ini **bukan annual turnover rate**, karena tidak menggunakan rata-rata headcount selama periode tahunan.

In [ ]:
query_department = """
SELECT
    Department,
    COUNT(DISTINCT EmpID) AS Total_Employees,
    SUM(CASE WHEN Termd = 0 THEN 1 ELSE 0 END) AS Not_Terminated,
    SUM(CASE WHEN Termd = 1 THEN 1 ELSE 0 END) AS Terminated,
    ROUND(
        100.0 * SUM(CASE WHEN Termd = 1 THEN 1 ELSE 0 END)
        / NULLIF(COUNT(*), 0), 2
    ) AS Termination_Share_Pct
FROM employees
GROUP BY Department
ORDER BY Total_Employees DESC;
"""

department_summary = pd.read_sql_query(query_department, conn)
display(department_summary)

,Department,Total_Employees,Not_Terminated,Terminated,Termination_Share_Pct
0,Production,209,126,83,39.71
1,IT/IS,50,40,10,20.00
2,Sales,31,26,5,16.13
3,Software Engineering,11,7,4,36.36
4,Admin Offices,9,7,2,22.22
5,Executive Office,1,1,0,0.00


## 5. Hiring and Termination Patterns

Analisis perekrutan menghitung jumlah record berdasarkan tahun pada `DateofHire`. Grafik perekrutan dapat menunjukkan perubahan jumlah karyawan yang **tercatat dalam dataset**, tetapi tidak langsung mewakili semua aktivitas rekrutmen organisasi di masa lalu.

In [ ]:
query_hiring = """
SELECT
    CAST(strftime('%Y', DateofHire) AS INTEGER) AS Hire_Year,
    COUNT(*) AS Hires
FROM employees
WHERE DateofHire IS NOT NULL
GROUP BY CAST(strftime('%Y', DateofHire) AS INTEGER)
ORDER BY Hire_Year;
"""

hiring_by_year = pd.read_sql_query(query_hiring, conn)
display(hiring_by_year)

,Hire_Year,Hires
0,2006,1
1,2007,2
2,2008,3
3,2009,7
4,2010,9
5,2011,83
6,2012,45
7,2013,44
8,2014,60
9,2015,36


### 5.1 Termination Reasons

Alasan terminasi dianalisis hanya untuk pegawai dengan `Termd = 1`. Hasilnya menunjukkan alasan yang tercatat paling banyak, tanpa mengasumsikan sebab-akibat di luar informasi dataset.

In [ ]:
query_termination_reasons = """
SELECT
    COALESCE(NULLIF(TRIM(TermReason), ''), 'Not recorded') AS Termination_Reason,
    COUNT(*) AS Employee_Records
FROM employees
WHERE Termd = 1
GROUP BY COALESCE(NULLIF(TRIM(TermReason), ''), 'Not recorded')
ORDER BY Employee_Records DESC, Termination_Reason;
"""

termination_reasons = pd.read_sql_query(query_termination_reasons, conn)
display(termination_reasons.head(15))

,Termination_Reason,Employee_Records
0,Another position,20
1,unhappy,14
2,more money,11
3,career change,9
4,hours,8
5,attendance,7
6,relocation out of area,5
7,return to school,5
8,military,4
9,"no-call, no-show",4


## 6. Employee Performance

Distribusi kinerja dihitung berdasarkan `PerformanceScore`. Kategori penilaian diperlakukan sebagai label, bukan angka berjarak sama.

Persentase pada setiap kategori dihitung terhadap seluruh pegawai dalam dataset.

In [ ]:
query_performance = """
SELECT
    PerformanceScore,
    COUNT(*) AS Employees,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM employees), 2)
        AS Share_Pct
FROM employees
GROUP BY PerformanceScore
ORDER BY Employees DESC;
"""

performance_summary = pd.read_sql_query(query_performance, conn)
display(performance_summary)

,PerformanceScore,Employees,Share_Pct
0,Fully Meets,243,78.14
1,Exceeds,37,11.90
2,Needs Improvement,18,5.79
3,PIP,13,4.18


### 6.1 Performance Mix by Department

Jumlah pegawai dengan setiap kategori performa dibandingkan dengan total pegawai pada departemen yang sama. Persentase ini lebih mudah dibandingkan daripada jumlah absolut karena ukuran departemen tidak sama.

In [ ]:
query_performance_dept = """
SELECT
    Department,
    PerformanceScore,
    COUNT(*) AS Employees,
    ROUND(
        100.0 * COUNT(*) /
        NULLIF((SELECT COUNT(*)
                FROM employees e2
                WHERE e2.Department = e.Department), 0),
        2
    ) AS Department_Share_Pct
FROM employees e
GROUP BY Department, PerformanceScore
ORDER BY Department, Employees DESC;
"""

performance_by_department = pd.read_sql_query(
    query_performance_dept, conn
)
display(performance_by_department)

,Department,PerformanceScore,Employees,Department_Share_Pct
0,Admin Offices,Fully Meets,9,100.00
1,Executive Office,Fully Meets,1,100.00
2,IT/IS,Fully Meets,42,84.00
3,IT/IS,Exceeds,6,12.00
4,IT/IS,PIP,1,2.00
5,IT/IS,Needs Improvement,1,2.00
6,Production,Fully Meets,159,76.08
7,Production,Exceeds,27,12.92
8,Production,Needs Improvement,15,7.18
9,Production,PIP,8,3.83


## 7. Engagement and Satisfaction

Rata-rata `EngagementSurvey` dan `EmpSatisfaction` dirangkum per departemen. Jumlah pegawai selalu ditampilkan untuk membantu menginterpretasikan hasil pada departemen dengan ukuran sampel kecil.

Perbedaan rata-rata merupakan temuan deskriptif, bukan bukti penyebab perbedaan kinerja.

In [ ]:
query_engagement = """
SELECT
    Department,
    COUNT(*) AS Employees,
    ROUND(AVG(EngagementSurvey), 2) AS Avg_Engagement,
    ROUND(AVG(EmpSatisfaction), 2) AS Avg_Satisfaction
FROM employees
GROUP BY Department
ORDER BY Avg_Engagement DESC;
"""

engagement_by_department = pd.read_sql_query(query_engagement, conn)
display(engagement_by_department)

,Department,Employees,Avg_Engagement,Avg_Satisfaction
0,Executive Office,1,4.83,3.00
1,Admin Offices,9,4.39,3.56
2,IT/IS,50,4.15,3.96
3,Production,209,4.13,3.86
4,Software Engineering,11,4.06,4.09
5,Sales,31,3.82,4.03


## 8. Attendance and Performance

Absensi (`Absences`) dan keterlambatan pada 30 hari terakhir (`DaysLateLast30`) dibandingkan berdasarkan kategori kinerja. Hasilnya berupa gambaran deskriptif. Hubungan antara kedua indikator tidak menunjukkan sebab-akibat.

In [ ]:
query_attendance = """
SELECT
    PerformanceScore,
    COUNT(*) AS Employees,
    ROUND(AVG(Absences), 2) AS Avg_Absences,
    ROUND(AVG(DaysLateLast30), 2) AS Avg_Late_Last30
FROM employees
GROUP BY PerformanceScore
ORDER BY Avg_Late_Last30 DESC;
"""

attendance_by_performance = pd.read_sql_query(query_attendance, conn)
display(attendance_by_performance)

,PerformanceScore,Employees,Avg_Absences,Avg_Late_Last30
0,PIP,13,8.31,4.31
1,Needs Improvement,18,11.33,3.78
2,Fully Meets,243,10.22,0.02
3,Exceeds,37,10.49,0.00


### 8.1 Attendance by Department

Analisis tambahan ini membantu menemukan departemen dengan rata-rata absensi dan keterlambatan relatif tinggi. Temuan perlu dibaca bersama jumlah pegawai dan tidak digunakan sebagai penilaian individu.

In [ ]:
query_attendance_department = """
SELECT
    Department,
    COUNT(*) AS Employees,
    ROUND(AVG(Absences), 2) AS Avg_Absences,
    ROUND(AVG(DaysLateLast30), 2) AS Avg_Late_Last30
FROM employees
GROUP BY Department
ORDER BY Avg_Late_Last30 DESC;
"""

attendance_by_department = pd.read_sql_query(
    query_attendance_department, conn
)
display(attendance_by_department)

,Department,Employees,Avg_Absences,Avg_Late_Last30
0,Sales,31,11.55,0.65
1,Production,209,10.14,0.47
2,Software Engineering,11,8.73,0.36
3,IT/IS,50,10.44,0.14
4,Executive Office,1,10.00,0.00
5,Admin Offices,9,8.67,0.00


## 9. Data Quality Checks in SQL

Sebagai pemeriksaan silang, SQL digunakan untuk menghitung `ManagerID` yang kosong serta record dengan tanggal penilaian kinerja sebelum tanggal perekrutan.

`DateofTermination` yang kosong pada record `Termd = 0` **bukan masalah data**. Koreksi DOB pada notebook sebelumnya harus tetap dirujuk dari log cleaning karena nilai asli sudah tidak tersedia dalam CSV hasil cleaning.

In [ ]:
query_quality = """
SELECT
    COUNT(*) AS Total_Employees,
    SUM(CASE WHEN ManagerID IS NULL THEN 1 ELSE 0 END)
        AS Missing_ManagerID,
    SUM(CASE
        WHEN LastPerformanceReview_Date IS NOT NULL
         AND DateofHire IS NOT NULL
         AND LastPerformanceReview_Date < DateofHire
        THEN 1 ELSE 0 END) AS Review_Before_Hire,
    SUM(CASE
        WHEN Termd = 0 AND DateofTermination IS NULL
        THEN 1 ELSE 0 END) AS Expected_Empty_Termination_Date
FROM employees;
"""

quality_sql_summary = pd.read_sql_query(query_quality, conn)
display(quality_sql_summary)

print("Catatan: Review_Before_Hire dan Missing_ManagerID adalah temuan review, bukan otomatis record yang harus dihapus.")

,Total_Employees,Missing_ManagerID,Review_Before_Hire,Expected_Empty_Termination_Date
0,311,8,2,207


Catatan: Review_Before_Hire dan Missing_ManagerID adalah temuan review, bukan otomatis record yang harus dihapus.


## 10. Validate SQL Results Against EDA

Perhitungan berikut memastikan angka utama SQL sesuai dengan jumlah record di CSV. Untuk dataset yang sama, hasil yang diharapkan adalah 311 pegawai, 207 belum terminasi, 104 telah terminasi, dan enam departemen.

Jika angka berbeda dari hasil EDA sebelumnya, hentikan proses dan periksa file input terlebih dahulu.

In [ ]:
total_sql = int(workforce_kpi.loc[0, "Total_Employees"])
active_sql = int(workforce_kpi.loc[0, "Not_Terminated"])
terminated_sql = int(workforce_kpi.loc[0, "Terminated"])

assert total_sql == df["EmpID"].nunique()
assert active_sql + terminated_sql == total_sql
assert int(department_summary["Total_Employees"].sum()) == total_sql
assert int(performance_summary["Employees"].sum()) == total_sql

print("SQL reconciliation passed.")
print(f"Employees: {total_sql}; Not terminated: {active_sql}; Terminated: {terminated_sql}")

SQL reconciliation passed.
Employees: 311; Not terminated: 207; Terminated: 104


## 11. Export SQL Results for Power BI

File CSV ringkasan hasil SQL disimpan untuk dokumentasi dan pemeriksaan ulang. Untuk dashboard utama, gunakan **`HRDataset_v14_clean_for_powerbi.csv`** agar filter dan hubungan antar-visual tetap fleksibel.

Tidak ada atribut tambahan pada dataset utama. Semua tabel ringkasan dibuat terpisah sebagai hasil query.

In [ ]:
exports = {
    "workforce_kpi.csv": workforce_kpi,
    "department_summary.csv": department_summary,
    "hiring_by_year.csv": hiring_by_year,
    "termination_reasons.csv": termination_reasons,
    "performance_summary.csv": performance_summary,
    "performance_by_department.csv": performance_by_department,
    "engagement_by_department.csv": engagement_by_department,
    "attendance_by_performance.csv": attendance_by_performance,
    "attendance_by_department.csv": attendance_by_department,
    "quality_sql_summary.csv": quality_sql_summary,
}

for filename, table in exports.items():
    table.to_csv(output_dir / filename, index=False)

df_sql.to_csv(
    output_dir / "HRDataset_v14_clean_for_powerbi.csv", index=False
)

# Menyimpan kumpulan query agar bisa diunggah ke GitHub.
queries = {
    "01_workforce_overview": query_workforce,
    "02_department_composition": query_department,
    "03_hiring_trend": query_hiring,
    "04_termination_reasons": query_termination_reasons,
    "05_performance_distribution": query_performance,
    "06_performance_by_department": query_performance_dept,
    "07_engagement_satisfaction": query_engagement,
    "08_attendance_by_performance": query_attendance,
    "09_attendance_by_department": query_attendance_department,
    "10_quality_checks": query_quality,
}

sql_text = "\n\n".join(
    f"-- {name}\n{query.strip()}"
    for name, query in queries.items()
)
(output_dir / "human_capital_analysis.sql").write_text(
    sql_text, encoding="utf-8"
)

print("Exported files:")
for file in sorted(output_dir.iterdir()):
    print("-", file.name)

Exported files:
- HRDataset_v14_clean_for_powerbi.csv
- attendance_by_department.csv
- attendance_by_performance.csv
- department_summary.csv
- engagement_by_department.csv
- hc_analytics.sqlite
- hiring_by_year.csv
- human_capital_analysis.sql
- performance_by_department.csv
- performance_summary.csv
- quality_sql_summary.csv
- termination_reasons.csv
- workforce_kpi.csv


### 11.1 Download the Analysis Outputs

Unduh ZIP untuk menyimpan database dan hasil query secara lokal. File ini dapat dipakai saat mengembangkan dashboard Power BI.

Untuk repository publik, utamakan notebook, `.sql`, screenshot hasil, dan dokumentasi. Perhatikan lisensi Kaggle sebelum mengunggah ulang CSV pegawai atau database yang berisi data turunannya.

In [ ]:
import shutil

archive = shutil.make_archive(
    "/content/HR_SQL_Analysis_Outputs", "zip", output_dir
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. SQL Analysis Findings

*Isi bagian ini setelah menjalankan seluruh query. Contoh struktur ringkasan berikut sengaja tidak mencantumkan kesimpulan yang belum diperiksa dari output.*

### Workforce Composition
- Total pegawai: [isi dari query].
- Departemen dengan jumlah pegawai terbesar: [nama departemen dan jumlah].
- Proporsi record terminasi tertinggi: [departemen, persentase, dan ukuran kelompok].

### Hiring and Termination
- Tahun perekrutan terbanyak: [tahun dan jumlah].
- Alasan terminasi yang paling sering tercatat: [alasan dan jumlah].

### Performance and Attendance
- Kategori kinerja terbesar: [kategori dan jumlah].
- Perbedaan rata-rata keterlambatan antar kategori: [ringkas hasil].

### Data Quality
- Missing ManagerID: [jumlah].
- Performance review sebelum tanggal masuk: [jumlah].
- Koreksi DOB: lihat `dob_correction_log.csv` dari notebook Data Cleaning.

### Interpretation Notes
- Dataset bersifat sintetis dan bersifat historis.
- Proporsi terminasi dalam dataset bukan annual turnover rate.
- Korelasi atau perbedaan kelompok tidak membuktikan hubungan sebab-akibat.
- Review findings harus didokumentasikan, bukan otomatis dihapus.

### Next Step
Hasil analisis akan digunakan untuk membuat tiga halaman Power BI: Employee Data Quality, Workforce Overview, dan Employee Performance & Attendance.